# 03 - Feature Engineering

**Author:** Juan Ruelas  
**Date:** 2026-05

We turn the cleaned hourly frame into a supervised-learning matrix. Four families of features are added, each motivated by what we saw in 02_eda:

1. **Calendar** - hour, day-of-week, month, weekend, German national holidays. These encode the human-behaviour drivers of demand.
2. **Lags** - 1, 2, 24, 48, 168 hours. Captures short-term persistence plus daily and weekly seasonality.
3. **Rolling stats** - 24h rolling mean and std. Smooths short-term noise and gives the model a sense of recent level and volatility.
4. **Interaction** - temperature x hour. Heating load is most temperature-sensitive in the evening; this term lets a tree model split on the combination directly.

Finally we split the frame chronologically: 2015-2018 -> train, 2019 -> test. Shuffling would leak future information through lag features, so we keep it strictly time-ordered.

In [1]:
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

processed_dir = Path('..') / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(processed_dir / 'merged.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print('input shape:', df.shape)
df.head()

input shape: (43824, 6)


,timestamp,load_mw,temperature_2m,relative_humidity_2m,wind_speed_10m,direct_radiation
0,2015-01-01 00:00:00,41151.0,3.8,96,14.4,0.0
1,2015-01-01 01:00:00,40135.0,3.6,95,14.9,0.0
2,2015-01-01 02:00:00,39106.0,3.3,94,14.6,0.0
3,2015-01-01 03:00:00,38765.0,3.0,94,14.1,0.0
4,2015-01-01 04:00:00,38941.0,2.3,94,13.0,0.0


## 1. Calendar features

We use Germany's national holiday calendar via the `holidays` package. Regional holidays (e.g. Reformation Day in some Lander) are intentionally excluded - they would inflate dimensionality without buying much because they affect only a fraction of national load.

In [2]:
de_holidays = holidays.country_holidays('DE', years=range(2015, 2021))

df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_holiday'] = df['timestamp'].dt.date.map(lambda d: int(d in de_holidays))

df[['timestamp', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday']].head(8)

,timestamp,hour,day_of_week,month,is_weekend,is_holiday
0,2015-01-01 00:00:00,0,3,1,0,1
1,2015-01-01 01:00:00,1,3,1,0,1
2,2015-01-01 02:00:00,2,3,1,0,1
3,2015-01-01 03:00:00,3,3,1,0,1
4,2015-01-01 04:00:00,4,3,1,0,1
5,2015-01-01 05:00:00,5,3,1,0,1
6,2015-01-01 06:00:00,6,3,1,0,1
7,2015-01-01 07:00:00,7,3,1,0,1


## 2. Lag features

Why these specific lags:

- **1h, 2h** - persistence; load barely changes minute-to-minute, so the previous hour is by far the strongest single predictor.
- **24h, 48h** - same hour yesterday and the day before. Captures daily seasonality robustly.
- **168h** - same hour same weekday a week ago. Strongest predictor that respects weekday-vs-weekend structure.

Lags introduce NaNs at the start of the series; we drop those rows once all features are built.

In [3]:
for lag in [1, 2, 24, 48, 168]:
    df[f'load_lag_{lag}h'] = df['load_mw'].shift(lag)

## 3. Rolling statistics

We use `shift(1)` before rolling so that the value at time t only sees information up to t-1 - this avoids label leakage. A 24h window is a natural choice because it averages over one full daily cycle.

In [4]:
shifted = df['load_mw'].shift(1)
df['load_roll_mean_24h'] = shifted.rolling(window=24, min_periods=24).mean()
df['load_roll_std_24h'] = shifted.rolling(window=24, min_periods=24).std()

## 4. Interaction term

Heating sensitivity to outdoor temperature varies by time of day. A single multiplicative term `temperature_2m * hour` is a cheap surrogate that lets a tree model split on the joint variable without needing many one-hot-by-temperature splits.

In [5]:
df['temp_x_hour'] = df['temperature_2m'] * df['hour']

## 5. Drop NaNs from lags / rolls and check

In [6]:
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f'Dropped {before - len(df)} rows with NaN lag/rolling features')
print('Final shape:', df.shape)
df.head()

Dropped 168 rows with NaN lag/rolling features
Final shape: (43656, 19)


,timestamp,load_mw,temperature_2m,relative_humidity_2m,wind_speed_10m,direct_radiation,hour,day_of_week,month,is_weekend,is_holiday,load_lag_1h,load_lag_2h,load_lag_24h,load_lag_48h,load_lag_168h,load_roll_mean_24h,load_roll_std_24h,temp_x_hour
0,2015-01-08 00:00:00,48041.0,1.3,90,15.3,0.0,0,3,1,0,0,50460.0,54070.0,45125.0,45011.0,41151.0,60113.000000,9076.207612,0.0
1,2015-01-08 01:00:00,47074.0,1.0,90,15.6,0.0,1,3,1,0,0,48041.0,50460.0,44217.0,44015.0,40135.0,60234.500000,8884.334704,1.0
2,2015-01-08 02:00:00,47228.0,0.8,90,15.8,0.0,2,3,1,0,0,47074.0,48041.0,44368.0,44049.0,39106.0,60353.541667,8677.107873,1.6
3,2015-01-08 03:00:00,48253.0,1.0,90,16.1,0.0,3,3,1,0,0,47228.0,47074.0,45298.0,44730.0,38765.0,60472.708333,8465.074306,3.0
4,2015-01-08 04:00:00,51155.0,1.2,88,16.7,0.0,4,3,1,0,0,48253.0,47228.0,48598.0,46416.0,38941.0,60595.833333,8253.609944,4.8


## 6. Time-based train / test split

- **Train**: 2015-01-08 (first valid row after 168h warm-up) -> 2018-12-31  
- **Test** : 2019-01-01 -> 2019-12-31

In [7]:
train = df[df['timestamp'] < '2019-01-01'].copy()
test = df[df['timestamp'] >= '2019-01-01'].copy()

print('Train range:', train['timestamp'].min(), '->', train['timestamp'].max(), 'rows =', len(train))
print('Test  range:', test['timestamp'].min(), '->', test['timestamp'].max(), 'rows =', len(test))

Train range: 2015-01-08 00:00:00 -> 2018-12-31 23:00:00 rows = 34896
Test  range: 2019-01-01 00:00:00 -> 2019-12-31 23:00:00 rows = 8760


In [8]:
train_path = processed_dir / 'features_train.csv'
test_path  = processed_dir / 'features_test.csv'
train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)
print('saved', train_path)
print('saved', test_path)
print('feature cols:', [c for c in train.columns if c not in ('timestamp', 'load_mw')])

saved ..\data\processed\features_train.csv
saved ..\data\processed\features_test.csv
feature cols: ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'direct_radiation', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'load_lag_1h', 'load_lag_2h', 'load_lag_24h', 'load_lag_48h', 'load_lag_168h', 'load_roll_mean_24h', 'load_roll_std_24h', 'temp_x_hour']


**Summary.** We now have 17 features in addition to the target. The train/test split is leak-free (chronological, lags built before split) and ready for modelling in notebook 04.